# 03 · Simulador de transacciones en tiempo real

Envía operaciones con tarjeta nuevas, con la hora actual, al Eventstream `Bretema_ES`, que las deja en la tabla `transacciones` del Eventhouse. Además del tráfico normal inyecta cada pocos minutos:

- **Ráfagas de fraude** en tarjetas que pasaron por la Gasolineira Ponte Alta en octubre (3 a 5 compras online seguidas en comercios extranjeros de alto riesgo).
- **Compras en Portugal** de clientes de Pontevedra y Ourense.
- **Importes inusuales** en joyerías y tiendas de electrónica.
- **Intentos con tarjetas bloqueadas**, que salen denegados.

Cada escenario se anuncia en la salida de la celda, para que sepas qué buscar en KQL.

**Con una capacidad F2:** cambia el lenguaje del notebook a **Python** (no PySpark), que consume mucho menos, y detén la sesión al terminar. Si puedes, ejecuta el simulador en tu ordenador en lugar de aquí: es el mismo código (`simulador/simulador_transacciones.py`).

**La cadena de conexión es una clave.** Pégala solo en la celda de parámetros y no guardes el notebook con ella si lo vas a compartir.

In [ ]:
%pip install azure-eventhub --quiet

In [ ]:
# Parámetros
CONEXION = ""        # Eventstream › origen Endpoint personalizado › Event Hub › Claves › Cadena de conexión: clave principal
POR_SEGUNDO = 2      # operaciones normales por segundo (media)
DURACION_MIN = 15    # minutos de simulación (0 = sin límite; para con el botón de detener)
CADA_FRAUDE = 3      # minutos entre ráfagas de fraude
CADA_PORTUGAL = 2    # minutos entre compras en Portugal
CADA_IMPORTE = 4     # minutos entre importes inusuales
CADA_BLOQUEADA = 6   # minutos entre intentos con tarjetas bloqueadas
URL_REPO = "https://raw.githubusercontent.com/AbenFdez/Banco_Bretema_Prevencion_Fraude/main"

In [ ]:
# Descarga el simulador del repositorio y lo carga como módulo
import importlib.util
import sys
import urllib.request
from pathlib import Path

ruta = Path("/tmp/bretema/simulador_transacciones.py")
ruta.parent.mkdir(parents=True, exist_ok=True)
urllib.request.urlretrieve(f"{URL_REPO}/simulador/simulador_transacciones.py", ruta)
spec = importlib.util.spec_from_file_location("simulador_transacciones", ruta)
sim = importlib.util.module_from_spec(spec)
sys.modules["simulador_transacciones"] = sim
spec.loader.exec_module(sim)

referencia = sim.cargar_referencia(f"{URL_REPO}/datos", Path("/tmp/bretema/datos"))

In [ ]:
assert CONEXION.startswith("Endpoint="), "Pega la cadena de conexión del endpoint personalizado en la celda de parámetros"
sim.simular(
    sim.Eventstream(CONEXION, None),
    referencia,
    por_segundo=POR_SEGUNDO,
    duracion_min=DURACION_MIN,
    cada_fraude_min=CADA_FRAUDE,
    cada_portugal_min=CADA_PORTUGAL,
    cada_importe_min=CADA_IMPORTE,
    cada_bloqueada_min=CADA_BLOQUEADA,
    avisos=print,
)